# DNABERT2-E fine-tuning on hard-label training data

This notebook fine-tunes the repository's `EpigenDnabert2` classifier on the hard-label dataset stored in `Data/training_data/TrainingDataWithRejection_hg38_mincpg_4_minlen_10`.

It reuses the existing project code instead of reimplementing the training stack:
- `methyldl.data.dataset.SupervisedDataset` for parquet-backed sequence and CpG methylation loading.
- `methyldl.modelling.classifiers.dnabert2.EpigenDnabert2` for DNABERT2-E with methylation embeddings.
- `methyldl.modelling.classifiers.dnabert2.TrainingArguments` so the trainer setup matches the rest of the repository.

This version uses the DMR attention classification head, with `dmr_label` passed to the model as the contextual DMR identifier for each read.

The target labels are hard labels. In practice this dataset contains 39 biological cell types plus one rejection class.

In [1]:
import sys
from pathlib import Path
import json
import random
import gc

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from scipy.special import softmax
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
import transformers
from transformers import TrainerCallback, TrainerControl, TrainerState
from transformers.training_args import TrainingArguments as HFTrainingArguments
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "EDA":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from methyldl.data.dataset import SupervisedDataset
from methyldl.modelling.classifiers.dnabert2 import EpigenDnabert2, TrainingArguments

DATA_DIR = ROOT / "Data/training_data/TrainingDataWithRejection_hg38_mincpg_4_minlen_10"
FOUNDATION_MODEL_PATH = ROOT / "foundationalModels/DNABERT-2-117M"
LABELS_DICT_PATH = ROOT / "App/labels_dict.json"
OUTPUT_DIR = ROOT / "output/dnabert2_finetuning_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert DATA_DIR.exists(), f"Missing data directory: {DATA_DIR}"
assert FOUNDATION_MODEL_PATH.exists(), f"Missing foundation model directory: {FOUNDATION_MODEL_PATH}"


def has_triton_capable_gpu() -> bool:
    """Return whether the active CUDA device supports Triton attention."""
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability()
    return major >= 8


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_TRITON = has_triton_capable_gpu()

print(f"Project root: {ROOT}")
print(f"Device: {DEVICE}")
print(f"Triton attention enabled: {USE_TRITON}")
print(f"Output directory: {OUTPUT_DIR}")

%load_ext autoreload
%autoreload 2

Project root: /vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl
Device: cuda
Triton attention enabled: False
Output directory: /vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl/output/dnabert2_finetuning_notebook


In [2]:
SUMMARY_COLUMNS = ["label", "read_length", "dmr_label"]
summary_rows = []
label_series_by_split = {}
dmr_series_by_split = {}

for split in ("train", "valid", "test"):
    frame = pd.read_parquet(DATA_DIR / f"{split}.parquet", columns=SUMMARY_COLUMNS)
    label_series = frame["label"].astype(int)
    dmr_series = frame["dmr_label"].astype(int)
    label_series_by_split[split] = label_series
    dmr_series_by_split[split] = dmr_series
    summary_rows.append(
        {
            "split": split,
            "n_rows": len(frame),
            "n_unique_labels": int(label_series.nunique()),
            "max_label_id": int(label_series.max()),
            "missing_label_ids": sorted(set(range(int(label_series.max()) + 1)) - set(label_series.unique())),
            "n_unique_dmr_labels": int(dmr_series.nunique()),
            "max_dmr_label_id": int(dmr_series.max()),
            "read_length_mean": float(frame["read_length"].mean()),
            "read_length_p95": float(frame["read_length"].quantile(0.95)),
            "read_length_p99": float(frame["read_length"].quantile(0.99)),
            "read_length_max": int(frame["read_length"].max()),
        }
    )

summary_df = pd.DataFrame(summary_rows).set_index("split")
display(summary_df)

with open(LABELS_DICT_PATH) as handle:
    labels_dict = {int(key): value for key, value in json.load(handle).items()}

num_labels = max(int(series.max()) for series in label_series_by_split.values()) + 1
num_dmr_labels = max(int(series.max()) for series in dmr_series_by_split.values()) + 1
if num_labels - 1 not in labels_dict:
    labels_dict[num_labels - 1] = "rejection"

label_names = [labels_dict.get(label_id, f"class_{label_id}") for label_id in range(num_labels)]
label_count_table = pd.DataFrame({
    split: label_series_by_split[split].value_counts().sort_index() for split in label_series_by_split
}).reindex(range(num_labels), fill_value=0)
label_count_table.index.name = "label_id"
label_count_table.insert(0, "label_name", label_names)

dmr_summary_df = pd.DataFrame(
    {
        split: {
            "n_unique_dmr_labels": int(dmr_series_by_split[split].nunique()),
            "max_dmr_label_id": int(dmr_series_by_split[split].max()),
        }
        for split in dmr_series_by_split
    }
).T

display(label_count_table)
display(dmr_summary_df)
print(f"Total number of target classes used for training: {num_labels}")
print(f"Total number of DMR ids reserved by the attention head: {num_dmr_labels}")

,n_rows,n_unique_labels,max_label_id,missing_label_ids,n_unique_dmr_labels,max_dmr_label_id,read_length_mean,read_length_p95,read_length_p99,read_length_max
split,,,,,,,,,,
train,132485,40,39,[],843,965,118.986051,150.0,158.0,351
valid,59274,39,39,[28],844,965,119.361879,150.0,158.0,315
test,4445834,39,39,[35],844,965,120.801009,150.0,162.0,448


,label_name,train,valid,test
label_id,,,,
0,Thyroid-Ep,970,990.0,835.0
1,Blood-Granul,530,852.0,736.0
2,Heart-Cardio,2430,1508.0,2669.0
3,Lung-Ep-Alveo,894,649.0,860.0
4,Kidney-Ep,3661,1399.0,963.0
5,Neuron,4375,1710.0,2532.0
6,Liver-Hep,4102,1058.0,1042.0
7,Dermal-Fibro,262,87.0,88.0
8,Bladder-Ep,1574,324.0,550.0


,n_unique_dmr_labels,max_dmr_label_id
train,843,965
valid,844,965
test,844,965


Total number of target classes used for training: 40
Total number of DMR ids reserved by the attention head: 966


## Training configuration

The train and validation splits are used in full by default.

The test split contains more than four million reads, so the notebook evaluates on a stratified sample by default to keep iteration practical. Set `TEST_EVAL_SAMPLE_N = None` later in the notebook if you want to score the full test split.

`lazy_tokenization=True` is used when building datasets so the notebook does not materialize tokenized tensors for every read up front.

In [7]:
SEED = 8
TRAIN_SAMPLE_N = 96000*2
VALID_SAMPLE_N = 10_000
TEST_EVAL_SAMPLE_N = 100
MAX_SEQUENCE_LENGTH = 150
LEARNING_RATE = 1e-5
NUM_TRAIN_EPOCHS = 1
PER_DEVICE_TRAIN_BATCH_SIZE = 800 if DEVICE == "cuda" else 2
PER_DEVICE_EVAL_BATCH_SIZE = 800 if DEVICE == "cuda" else 2
LR_SCHEDULER_TYPE = transformers.SchedulerType.COSINE
GRADIENT_ACCUMULATION_STEPS = 1
SAVE_STEPS = 20
EVAL_STEPS = 1
LOGGING_STEPS = EVAL_STEPS
ALIBI_SLOPE_MULTIPLIER = 1.0
POSITIONAL_ENCODING = "rope"
WARMUP_STEPS = 50
SAVE_TOTAL_LIMIT = 2
DMR_LABEL_COLUMN = "dmr_label"
DATASET_COLUMNS = ["seq", "pattern", "label", DMR_LABEL_COLUMN]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


def stratified_sample(frame: pd.DataFrame, n_samples: int | None, seed: int = SEED) -> pd.DataFrame:
    """Return a stratified sample of a labeled dataframe."""
    if n_samples is None or n_samples >= len(frame):
        return frame.reset_index(drop=True)
    sampled, _ = train_test_split(
        frame,
        train_size=n_samples,
        stratify=frame["label"],
        random_state=seed,
    )
    return sampled.reset_index(drop=True)


training_args = TrainingArguments(
    run_name="dnabert2_hard_labels_notebook",
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=PER_DEVICE_EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_TRAIN_EPOCHS,
    warmup_steps=WARMUP_STEPS,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    eval_steps=EVAL_STEPS,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    # overwrite_output_dir=True,
    fp16=(DEVICE == "cuda"),
    report_to=[],
    remove_unused_columns=False,
    prediction_loss_only=False,
    skip_memory_metrics=True,
    auto_find_batch_size=False,
    lr_scheduler_type=LR_SCHEDULER_TYPE
)

# print(training_args)

In [8]:
def load_split_frame(split: str, columns: list[str], n_samples: int | None) -> pd.DataFrame:
    """Load one parquet split with optional class-preserving sampling."""
    split_path = DATA_DIR / f"{split}.parquet"
    if n_samples is None:
        return pd.read_parquet(split_path, columns=columns).reset_index(drop=True)

    if split != "test":
        frame = pd.read_parquet(split_path, columns=columns)
        return stratified_sample(frame, n_samples)

    # For the test split, sample an equal number of reads from every class so
    # that the evaluation is balanced across all cell types.
    test_counts = label_count_table["test"].fillna(0).astype(int)
    available_classes = test_counts[test_counts > 0]
    n_per_class = n_samples // len(available_classes)

    sampled_parts = []
    for label_id, available in available_classes.items():
        target = min(n_per_class, int(available))
        if target <= 0:
            continue
        # Read one label at a time to avoid materialising the full test split.
        part = pd.read_parquet(
            split_path,
            columns=columns,
            filters=[("label", "=", int(label_id))],
        )
        if len(part) > target:
            part = part.sample(n=target, random_state=SEED)
        sampled_parts.append(part)

    sampled = pd.concat(sampled_parts, ignore_index=True)
    return sampled.sample(frac=1.0, random_state=SEED).reset_index(drop=True)


train_df = load_split_frame("train", DATASET_COLUMNS, TRAIN_SAMPLE_N)
valid_df = load_split_frame("valid", DATASET_COLUMNS, VALID_SAMPLE_N)
test_eval_df = load_split_frame("test", DATASET_COLUMNS, TEST_EVAL_SAMPLE_N)

model = EpigenDnabert2(
    foundation_model_huggingface=str(FOUNDATION_MODEL_PATH),
    max_sequence_length=MAX_SEQUENCE_LENGTH,
    num_labels=num_labels,
    num_dmr_labels=num_dmr_labels,
    use_cpg_methylation=True,
    use_m6a_methylation=False,
    use_triton=USE_TRITON,
    training_args=training_args,
    alibi_slope_multiplier=ALIBI_SLOPE_MULTIPLIER,
    positional_encoding=POSITIONAL_ENCODING,
)
print(f"Using DMR attention head with {num_dmr_labels} reserved DMR ids.")

train_dataset = SupervisedDataset(
    data_path_or_list=train_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)
valid_dataset = SupervisedDataset(
    data_path_or_list=valid_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)
test_eval_dataset = SupervisedDataset(
    data_path_or_list=test_eval_df,
    tokenizer=model.tokenizer,
    data_interface="pandas",
    lazy_tokenization=True,
    include_dmr_ids=True,
    dmr_label_column=DMR_LABEL_COLUMN,
)

display(
    pd.DataFrame(
        [
            {"split": "train", "rows_used": len(train_df)},
            {"split": "valid", "rows_used": len(valid_df)},
            {"split": "test_eval", "rows_used": len(test_eval_df)},
        ]
    )
)


/data/leuven/389/vsc38912/.cache/huggingface/modules/transformers_modules/DNABERT_hyphen_2_hyphen_117M/bert_layers.py:193: UserWarning: Triton is disabled by config, using default self-attention implementation
  warnings.warn(


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl/foundationalModels/DNABERT-2-117M
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: /vsc-hard-mounts/leuven-data/389/vsc38912/Projects/methyldl/foundationalModels/DNABERT-2-117M
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.bias                     | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were 

Using DMR attention head with 966 reserved DMR ids.


,split,rows_used
0,train,132485
1,valid,10000
2,test_eval,78


In [9]:
test_eval_dataset

In [10]:
model.predict(test_eval_dataset, batch_size=5)

[RANK 0] Detected kernel version 4.18.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.
`use_return_dict` is deprecated! Use `return_dict` instead!


PredictionOutput(predictions=array([[nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       ...,
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan],
       [nan, nan, nan, ..., nan, nan, nan]], shape=(78, 40), dtype=float32), label_ids=array([20, 18, 36,  1, 23, 22,  0,  6,  8,  9, 39, 16, 22, 37, 38, 16, 29,
       11, 32, 17,  5, 39, 12, 26, 12, 28, 15,  3, 25, 34, 27,  0,  5, 19,
       17, 27, 38, 10, 10, 15, 11, 36, 14, 33, 21,  8, 23, 18, 32, 31,  1,
       13, 20, 21,  3, 37, 26,  2, 25, 34, 28, 14, 19,  7,  9, 29,  7, 30,
        4,  6, 13, 24, 30, 24, 31,  4,  2, 33]), metrics={'test_loss': nan, 'test_model_preparation_time': 0.0023, 'test_runtime': 0.5863, 'test_samples_per_second': 133.038, 'test_steps_per_second': 27.29})

In [ ]:
# add callback to track weight norms during training
class WeightNormCallback(TrainerCallback):
    """Record global weight norm and per-role (Q/K/V) attention norms at every logging step.

    DNABERT-2 uses a fused Wqkv projection (shape: 3*H x H) inside each
    BertUnpadSelfAttention layer.  The three blocks are stacked row-wise:
        Wqkv.weight[ :H , :]  → query projection
        Wqkv.weight[H:2H, :]  → key   projection
        Wqkv.weight[2H: , :]  → value projection

    All norms are Frobenius norms accumulated across every attention layer
    in the network, so they reflect the combined magnitude of all Q / K / V
    matrices.
    """

    def __init__(self, tracked_model: torch.nn.Module) -> None:
        self.tracked_model = tracked_model
        self.history: list[dict] = []

    def on_log(
        self,
        args: HFTrainingArguments,
        state: TrainerState,
        control: TrainerControl,
        logs: dict | None = None,
        **kwargs,
    ) -> None:
        # Only record at training steps (the log dict contains "loss").
        if not logs or "loss" not in logs:
            return

        total_sq = 0.0
        q_sq = k_sq = v_sq = 0.0

        for name, param in self.tracked_model.named_parameters():
            if not param.requires_grad:
                continue
            data = param.detach().float()
            total_sq += data.norm() ** 2

            # Fused QKV weight: name ends with "Wqkv.weight"
            if name.endswith("Wqkv.weight"):
                H = data.shape[0] // 3
                q_sq += data[:H].norm() ** 2
                k_sq += data[H : 2 * H].norm() ** 2
                v_sq += data[2 * H :].norm() ** 2

        self.history.append(
            {
                "step": state.global_step,
                "weight_norm": float(total_sq**0.5),
                "q_norm": float(q_sq**0.5),
                "k_norm": float(k_sq**0.5),
                "v_norm": float(v_sq**0.5),
            }
        )


weight_norm_callback = WeightNormCallback(model.model)

In [ ]:
gc.collect()
train_result = model.fine_tune(
    training_args=training_args,
    train_dataset=train_dataset,
    val_dataset=valid_dataset,
    test_dataset=test_eval_dataset,
    callbacks=[weight_norm_callback],
)
train_result

In [ ]:
# Compute the norm of all weights in the model
total_norm = 0.0
for param in model.model.parameters():
    if param.requires_grad:
        total_norm += param.data.norm() ** 2

total_norm = float(total_norm ** 0.5)
print(f"Total weight norm of the model: {total_norm:.6f}")

In [ ]:
import matplotlib.pyplot as plt

log_history = model.trainer.state.log_history

train_entries = [e for e in log_history if "loss" in e and "eval_loss" not in e]
eval_entries  = [e for e in log_history if "eval_loss" in e]

steps         = [e["step"]              for e in train_entries]
grad_norm     = [e.get("grad_norm")     for e in train_entries]
learning_rate = [e.get("learning_rate") for e in train_entries]
epoch         = [e.get("epoch")         for e in train_entries]
loss          = [e["loss"]              for e in train_entries]
eval_steps    = [e["step"]              for e in eval_entries]
eval_loss     = [e["eval_loss"]         for e in eval_entries]

fig, ax1 = plt.subplots(figsize=(14, 6))
fig.suptitle("Training log history", fontsize=13)

# Losses share the left y-axis
l1, = ax1.plot(steps,      loss,      marker="+", color="tab:blue",   lw=1.5, label="loss")
l2, = ax1.plot(eval_steps, eval_loss, marker="x", color="tab:orange", lw=1.5, linestyle="--", label="eval_loss")
ax1.set_xlabel("step")
ax1.set_ylabel("loss")
ax1.tick_params(axis="y", labelcolor="tab:blue")

# grad_norm on a second y-axis (right)
ax2 = ax1.twinx()
l3, = ax2.plot(steps, grad_norm, marker="o", color="tab:green", lw=1.2, alpha=0.8, label="grad_norm")
ax2.set_ylabel("grad_norm", color="tab:green")
ax2.tick_params(axis="y", labelcolor="tab:green")

# learning_rate on a third y-axis (further right, offset)
ax3 = ax1.twinx()
ax3.spines["right"].set_position(("axes", 1.10))
l4, = ax3.plot(steps, learning_rate, marker="s", color="tab:red", lw=1.2, alpha=0.8, label="learning_rate")
ax3.set_ylabel("learning_rate", color="tab:red")
ax3.tick_params(axis="y", labelcolor="tab:red")

lines = [l1, l2, l3, l4]
ax1.legend(lines, [l.get_label() for l in lines], loc="upper right")
ax1.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()

plot_name = (
    f"train{TRAIN_SAMPLE_N}"
    f"_val{VALID_SAMPLE_N}"
    f"_bs{PER_DEVICE_TRAIN_BATCH_SIZE}"
    f"_lr{LEARNING_RATE}"
    f"_warmup{WARMUP_STEPS}"
    f"_sched{LR_SCHEDULER_TYPE.value}"
    f"_hardlabels"
    f"_seqlen{MAX_SEQUENCE_LENGTH}"
    f"_posenc{POSITIONAL_ENCODING}"
    f"_alibi{ALIBI_SLOPE_MULTIPLIER}"
    ".png"
)
plot_path = OUTPUT_DIR / plot_name
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
print(f"Plot saved to: {plot_path}")

plt.show()

In [ ]:
# ── Weight-norm tracking ──────────────────────────────────────────────────────
# WeightNormCallback records global and per-role (Q/K/V) Frobenius norms after
# every logging step.  Norms are accumulated across ALL attention layers.

norm_df = pd.DataFrame(weight_norm_callback.history)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)
fig.suptitle("Weight norms during training", fontsize=13)

# Top panel: total weight norm of the network
ax1.plot(norm_df["step"], norm_df["weight_norm"], color="tab:blue", lw=1.5, marker="+")
ax1.set_ylabel("Global weight norm")
ax1.grid(True, linestyle="--", alpha=0.4)
ax1.set_title("Frobenius norm of all trainable parameters")

# Bottom panel: Q / K / V norms stacked together
ax2.plot(norm_df["step"], norm_df["q_norm"], color="tab:red",    lw=1.5, marker="o", label="Q")
ax2.plot(norm_df["step"], norm_df["k_norm"], color="tab:green",  lw=1.5, marker="s", label="K")
ax2.plot(norm_df["step"], norm_df["v_norm"], color="tab:purple", lw=1.5, marker="^", label="V")
ax2.set_xlabel("step")
ax2.set_ylabel("Frobenius norm (all layers combined)")
ax2.set_title("Frobenius norm of Q / K / V projection weights (summed across all attention layers)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / f"weight_norms_{plot_name}", dpi=150, bbox_inches="tight")
plt.show()

display(norm_df.set_index("step").describe())


NEXT STEPS: INVESTIGATE IF THERE IS A DISCREPANCY IN PERFORMANCE BETWEEN THE CLASSES:

Yes there is: some classes are predicted almost always in rejection, while others are predicted almost always in their own class.


In [ ]:
def top_confusions(confusion: np.ndarray, top_n: int = 15) -> pd.DataFrame:
    """Return the largest off-diagonal entries from a confusion matrix."""
    rows = []
    for true_idx in range(confusion.shape[0]):
        for pred_idx in range(confusion.shape[1]):
            if true_idx == pred_idx:
                continue
            count = int(confusion[true_idx, pred_idx])
            if count == 0:
                continue
            rows.append(
                {
                    "true_label_id": true_idx,
                    "true_label_name": label_names[true_idx],
                    "pred_label_id": pred_idx,
                    "pred_label_name": label_names[pred_idx],
                    "count": count,
                }
            )
    return pd.DataFrame(rows).sort_values("count", ascending=False).head(top_n)


def evaluate_split(split_name: str, dataset: SupervisedDataset):
    """Predict one split and return summary metrics plus error-analysis tables."""
    prediction_output = model.predict(dataset)
    logits = np.asarray(prediction_output.predictions)
    y_true = np.asarray(prediction_output.label_ids).astype(int)

    if logits.ndim == 1:
        probabilities = logits
        y_pred = (probabilities >= 0.5).astype(int)
    else:
        # The model returns raw multi-class logits here, so convert them to
        # probabilities before taking the most likely class for evaluation.
        probabilities = softmax(logits, axis=1)
        y_pred = probabilities.argmax(axis=1)

    metrics = {
        "split": split_name,
        "n_samples": int(len(y_true)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
    }

    report = pd.DataFrame(
        classification_report(
            y_true,
            y_pred,
            labels=list(range(num_labels)),
            target_names=label_names,
            output_dict=True,
            zero_division=0,
        )
    ).T
    confusion = confusion_matrix(y_true, y_pred, labels=list(range(num_labels)))
    confusion_df = pd.DataFrame(confusion, index=label_names, columns=label_names)

    return metrics, report, confusion_df, top_confusions(confusion)


valid_metrics, valid_report, valid_confusions, valid_top_confusions = evaluate_split("valid", valid_dataset)
test_metrics, test_report, test_confusions, test_top_confusions = evaluate_split("test", test_eval_dataset)

display(pd.DataFrame([valid_metrics, test_metrics]).set_index("split"))
display(valid_top_confusions)
display(test_top_confusions)

In [ ]:
def plot_confusion_matrix(
    confusions_df: pd.DataFrame,
    title: str = "Confusion matrix",
    save_path: Path | None = None,
    row_normalized: bool = True,
) -> None:
    """Plot a confusion matrix heatmap.

    Parameters
    ----------
    confusions_df : pd.DataFrame
        Square confusion matrix with label names as both index and columns.
    title : str
        Plot title.
    save_path : Path or None
        If provided, save the figure to this path.
    row_normalized : bool
        If True, plot row-normalized proportions. If False, plot raw counts.
    """
    cm = confusions_df.to_numpy(dtype=float)
    labels = confusions_df.index.tolist()

    if row_normalized:
        row_sums = cm.sum(axis=1, keepdims=True)
        values = np.divide(cm, row_sums, out=np.zeros_like(cm), where=row_sums != 0)
        cmap_label = "Proportion within true class"
    else:
        values = cm
        cmap_label = "Count"

    fig, ax = plt.subplots(figsize=(16, 14))
    im = ax.imshow(values, cmap="Blues", vmin=0, vmax=1 if row_normalized else None)

    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=90, fontsize=7)
    ax.set_yticklabels(labels, fontsize=7)

    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label(cmap_label)

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"Plot saved to: {save_path}")

    plt.show()


plot_confusion_matrix(valid_confusions, title="Validation confusion matrix (row-normalized)")
plot_confusion_matrix(test_confusions, title="Test confusion matrix (row-normalized)")

plot_confusion_matrix(
    valid_confusions,
    title="Validation confusion matrix (raw counts)",
    row_normalized=False,
)
plot_confusion_matrix(
    test_confusions,
    title="Test confusion matrix (raw counts)",
    row_normalized=False,
)

In [ ]:
# now we plot a scatter plot of the accuracy versus the class count in the training set, to see if there is a correlation between the two
accuracy_by_class = valid_report.loc[label_names, "precision"].fillna(0)
class_counts = label_count_table["train"].fillna(0).astype(int)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(class_counts, accuracy_by_class, alpha=0.7)
ax.set_xlabel("Number of training samples for class")
ax.set_ylabel("Validation precision")
ax.set_title("Validation precision vs. training class count")
plt.xlim(0, 10000)
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# scatter plot of validation precision vs. region mcc rank 
best_to_worse_cell_by_mcc_in_region = [
    "Small-Int-Ep", "Heart-Cardio", "Thyroid-Ep", "Gastric-Ep", "Pancreas-Acinar",
    "Blood-Granul", "Eryth-prog", "Blood-B", "Colon-Ep", "Oligodend",
    "Lung-Ep-Alveo", "Lung-Ep-Bron", "Pancreas-Beta", "Pancreas-Alpha", "Liver-Hep",
    "Pancreas-Delta", "Bladder-Ep", "Gallbladder", "Fallopian-Ep", "Blood-NK",
    "Breast-Luminal-Ep", "Neuron", "Kidney-Ep", "Head-Neck-Ep", "Pancreas-Duct",
    "Breast-Basal-Ep", "Blood-T", "Prostate-Ep", "Epid-Kerat", "Bone-Osteob",
    "Heart-Fibro", "Blood-Mono+Macro", "Adipocytes", "Skeletal-Musc", "Dermal-Fibro",
    "Endothel", "Smooth-Musc", "Ovary-Ep", "Colon-Fibro"
]
accuracy_by_class = valid_report.loc[label_names, "precision"].fillna(0)
mcc_rank_by_class = pd.Series(
    data=range(len(best_to_worse_cell_by_mcc_in_region)),
    index=best_to_worse_cell_by_mcc_in_region,
    name="mcc_rank"
)
scatter_df = pd.DataFrame({
    "accuracy": accuracy_by_class,
    "mcc_rank": mcc_rank_by_class,
}).dropna(subset=["accuracy", "mcc_rank"])
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(scatter_df["mcc_rank"], scatter_df["accuracy"], alpha=0.7)
ax.set_xlabel("Cell type rank by region MCC")
ax.set_ylabel("Validation precision")
ax.set_title("Validation precision vs. cell type rank by region MCC")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# val accuracy vs test accuracy per class
accuracy_by_class_valid = valid_report.loc[label_names, "precision"].fillna(0)
accuracy_by_class_test = test_report.loc[label_names, "precision"].fillna(0)
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(accuracy_by_class_valid, accuracy_by_class_test, alpha=0.7)
ax.set_xlabel("Validation precision")
ax.set_ylabel("Test precision")
ax.set_title("Test precision vs. validation precision per class")
plt.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
test_report


In [ ]:
valid_report

## ALiBi as a suspect: does it handle short bisulfite reads correctly?

DNABERT-2 replaces sinusoidal/learned positional embeddings with **ALiBi (Attention with Linear Biases)**.  
ALiBi adds a head-specific linear penalty `slope_h × -|i − j|` to every attention score before softmax.

ALiBi was designed to help models **extrapolate to sequences longer** than those seen during pre-training.  
The concern for our setting is the opposite: DNABERT-2 was pre-trained on full-length genomic sequences  
(typically hundreds to thousands of nucleotides). Our bisulfite reads are 150 bp, which — after BPE  
tokenization — yields very few tokens.

When the sequence is short, **all ALiBi biases collapse into a very narrow numerical range**, so:
- Heads designed for long-range context (shallow slopes) become nearly indistinguishable from short-range heads (steep slopes).
- The model never encountered this compressed-bias regime during pre-training.
- The CLS token effectively sees near-isotropic attention from all positions — essentially no positional signal.

The cells below quantify and visualise this effect.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

# ── 1. ALiBi slope computation (mirrors bert_layers.py exactly) ───────────────

def _get_alibi_head_slopes(n_heads: int):
    def get_slopes_power_of_2(n: int):
        start = 2 ** (-(2 ** -(math.log2(n) - 3)))
        ratio = start
        return [start * ratio**i for i in range(n)]

    if math.log2(n_heads).is_integer():
        return get_slopes_power_of_2(n_heads)
    closest = 2 ** math.floor(math.log2(n_heads))
    slopes_a = get_slopes_power_of_2(closest)
    slopes_b = _get_alibi_head_slopes(2 * closest)
    slopes_b = slopes_b[0::2][: n_heads - closest]
    return slopes_a + slopes_b


def build_alibi_bias(n_heads: int, seq_len: int) -> np.ndarray:
    """Returns shape (n_heads, seq_len, seq_len) matching the model's bias."""
    slopes = np.array(_get_alibi_head_slopes(n_heads))
    positions = np.arange(seq_len)
    dist = np.abs(positions[:, None] - positions[None, :])          # (L, L)
    bias = slopes[:, None, None] * -dist[None, :, :]                 # (H, L, L)
    return bias


N_HEADS = model.model.bert.encoder.num_attention_heads  # 12 for DNABERT-2-117M

# ── 2. Measure actual token counts from a sample of train reads ───────────────

SAMPLE_SIZE = 500
sample_seqs = train_df["seq"].sample(n=SAMPLE_SIZE, random_state=SEED).tolist()
token_counts = [
    model.tokenizer(s, truncation=True, max_length=model.tokenizer.model_max_length)["input_ids"]
    for s in sample_seqs
]
token_lengths = [len(t) for t in token_counts]

print(f"Token count statistics over {SAMPLE_SIZE} training reads (150 bp):")
print(f"  min   : {min(token_lengths)}")
print(f"  median: {int(np.median(token_lengths))}")
print(f"  mean  : {np.mean(token_lengths):.1f}")
print(f"  max   : {max(token_lengths)}")
print()

# Tokens for a representative long genomic sequence vs. our short reads
L_short = int(np.median(token_lengths))     # typical read token count
L_pretrain = 128                            # plausible pre-training context length (tokens)

print(f"ALiBi bias range comparison:")
print(f"  Pre-training context ({L_pretrain} tokens):")
for name, slope in [("steepest head", max(_get_alibi_head_slopes(N_HEADS))),
                     ("shallowest head", min(_get_alibi_head_slopes(N_HEADS)))]:
    print(f"    {name:20s}  slope={slope:.5f}  max_penalty={slope * (L_pretrain - 1):.3f}")
print(f"  Fine-tuning reads  ({L_short} tokens):")
for name, slope in [("steepest head", max(_get_alibi_head_slopes(N_HEADS))),
                     ("shallowest head", min(_get_alibi_head_slopes(N_HEADS)))]:
    print(f"    {name:20s}  slope={slope:.5f}  max_penalty={slope * (L_short - 1):.3f}")


In [ ]:
# ── 3. Visualise how compressed the bias range is for short vs. long sequences ─

slopes = _get_alibi_head_slopes(N_HEADS)

fig, axes = plt.subplots(2, N_HEADS, figsize=(18, 5), sharex=False, sharey=False)
fig.suptitle("ALiBi bias matrices: pre-training length vs. short bisulfite read length", fontsize=12)

for h_idx, slope in enumerate(slopes):
    for row_idx, (L, label) in enumerate([(L_pretrain, f"pre-train\nL={L_pretrain}"),
                                           (L_short,    f"reads\nL={L_short}")]):
        ax = axes[row_idx, h_idx]
        dist = np.abs(np.arange(L)[:, None] - np.arange(L)[None, :])
        bias = slope * -dist
        im = ax.imshow(bias, cmap="RdBu", vmin=-slope * (L_pretrain - 1), vmax=0)
        ax.set_title(f"h{h_idx+1}\nslope={slope:.4f}", fontsize=5.5)
        ax.axis("off")
        if h_idx == 0:
            ax.set_ylabel(label, fontsize=7, rotation=0, labelpad=40, va="center")

fig.colorbar(im, ax=axes, fraction=0.01, pad=0.02, label="ALiBi bias")
plt.tight_layout()
plt.show()

# ── 4. Effective positional entropy: how uniform is attention after ALiBi? ─────
# Compute softmax over a single row of biases (without content similarity) to see
# how "flat" the positional distribution is.
#
# Raw entropy scales with log(L), so we normalise by log(L) to get a value in
# [0, 1] where 1 = perfectly uniform and 0 = fully concentrated.
# The ratio of *normalised* entropies tells us whether the head loses positional
# discrimination when moving from pre-training length to short read length.

def row_entropy_norm(slope: float, L: int) -> float:
    """Normalised Shannon entropy of ALiBi-only attention from the centre token."""
    dist = np.abs(np.arange(L) - (L // 2)).astype(float)
    logits = slope * -dist
    logits -= logits.max()
    probs = np.exp(logits) / np.exp(logits).sum()
    H = -np.sum(probs * np.log(probs + 1e-12))
    return H / math.log(L)   # normalise to [0, 1]


print("Normalised attention uniformity caused by ALiBi alone (1 = fully uniform, 0 = fully focused):")
print(f"{'Head':>5}  {'slope':>8}  {'norm_H(pre)':>12}  {'norm_H(reads)':>14}  {'ratio':>7}  interpretation")
for h_idx, slope in enumerate(slopes):
    H_pre_norm   = row_entropy_norm(slope, L_pretrain)
    H_short_norm = row_entropy_norm(slope, L_short)
    ratio = H_short_norm / H_pre_norm
    if ratio > 1.05:
        interp = "reads MORE uniform → positional focus degrades"
    elif ratio < 0.95:
        interp = "reads MORE focused  → head sharpens at shorter L"
    else:
        interp = "~stable"
    print(f"{h_idx+1:>5}  {slope:>8.5f}  {H_pre_norm:>12.3f}  {H_short_norm:>14.3f}  {ratio:>7.2f}  {interp}")

print()
print("norm_H close to 1 means the head is nearly uniform at that length (no positional signal).")
print("A ratio > 1 means the head loses positional focus on short reads vs. pre-training length.")
